# 🚀 Lessons 30-32: LoRA Fusion, GGUF Conversion & Ollama Deployment

**Advanced Step-by-Step Interactive Notebook** with clear architectural context, code logic, and step explanations.


# Lessons 30–32: QLoRA Fine-Tuning with SFTTrainer (Alpaca Dataset)

This notebook provides a streamlined, end-to-end implementation of **Quantized Low-Rank Adaptation (QLoRA)** using the Hugging Face `trl`, `peft`, `transformers`, and `bitsandbytes` libraries.

### Key Concepts Covered:
1. **8-Bit Quantization (BitsAndBytes)**: Loading weights in INT8 (`Linear8bitLt`) to reduce memory footprint by ~75% without performance loss.
2. **Baseline Pre-Training Inference**: Recording raw base model responses prior to instruction tuning.
3. **Custom Jinja Chat Template**: Injecting system/user/assistant token format and resizing embeddings for GPT-2.
4. **Dataset Preprocessing (Alpaca)**: Formatting instructions, inputs, and outputs into standard dialogue schemas.
5. **LoRA Adapters (`peft`)**: Injecting low-rank matrices ($r=8, \alpha=16$) into `c_attn`, `c_proj`, and `c_fc` linear layers.
6. **Supervised Fine-Tuning (`SFTTrainer`)**: Efficient multi-epoch training with sequence packing and BF16 precision.
7. **Evaluation & Generation**: Testing the fine-tuned model against pre-training baseline prompts.

## 1. Environment Setup & Library Imports

### 🔹 Step 1: Execution Block

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
# Install required libraries (uncomment if running in Google Colab or fresh environment)
# !pip install -q -U transformers datasets accelerate peft bitsandbytes trl

import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

## 2. Load 8-Bit Quantized Base Model (`BitsAndBytesConfig`)
We configure `load_in_8bit=True` using `BitsAndBytesConfig`. This quantizes the model weights to 8-bit integers while preserving 16-bit computation for activations, drastically cutting VRAM usage.

### 🔹 Step 2: Execution Block

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
MODEL_NAME = "gpt2"

# 1. Configure 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 3. Load Base Model with 8-bit Quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto"
)

# Check memory footprint
memory_mb = model.get_memory_footprint() / (1024 ** 2)
print(f"Model loaded successfully on {model.device}!")
print(f"Model Memory Footprint: {memory_mb:.2f} MB")

## 3. Pre-Training Baseline Inference
Before training, we evaluate the raw pre-trained base model on standard queries to establish a benchmark for comparison.

### 🔹 Step 3: Execution Block

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
def generate_response(prompt, max_new_tokens=50):
    """Generates raw completion from the base model without chat templates."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

baseline_prompts = [
    "What is the capital of France?",
    "What is 2 + 2?",
    "Explain what a computer is.",
    "Who wrote Romeo and Juliet?"
]

print("=" * 60)
print("RAW BASE MODEL RESPONSES (BEFORE INSTRUCTION TUNING)")
print("=" * 60)
for prompt in baseline_prompts:
    print(f"\n[Prompt]: {prompt}")
    print(f"[Output]: {generate_response(prompt)}")

## 4. Load Dataset & Configure Custom Jinja Chat Template
Since standard GPT-2 lacks a conversational chat template, we:
1. Add special role tokens (`<|system|>`, `<|user|>`, `<|assistant|>`) to the vocabulary and resize model embeddings.
2. Assign a custom Jinja chat template to `tokenizer.chat_template`.
3. Format the **tatsu-lab/alpaca** dataset.

### 🔹 Step 4: Execution Block

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
# 1. Add Special Role Tokens & Resize Embedding Matrix
special_tokens = {
    "additional_special_tokens": ["<|system|>", "<|user|>", "<|assistant|>"]
}
num_added = tokenizer.add_special_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))
print(f"Added {num_added} special tokens. New vocab size: {len(tokenizer)}")

# 2. Inject Custom Jinja Chat Template into Tokenizer
tokenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'system' %}<|system|>
{{ message['content'] }}
{% elif message['role'] == 'user' %}<|user|>
{{ message['content'] }}
{% elif message['role'] == 'assistant' %}<|assistant|>
{{ message['content'] }}
{% endif %}{% endfor %}{% if add_generation_prompt %}<|assistant|>
{% endif %}"""

# 3. Load Alpaca Dataset
dataset = load_dataset("tatsu-lab/alpaca")
print(f"Loaded Alpaca dataset with {len(dataset['train'])} training examples.")

# 4. Format Dataset Examples into Structured Chat Sequences
def format_alpaca_example(example):
    instruction = example["instruction"].strip()
    input_text = example.get("input", "").strip()
    output = example["output"].strip()

    if input_text:
        user_content = f"{instruction}\n\nInput:\n{input_text}"
    else:
        user_content = instruction

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

formatted_dataset = dataset["train"].map(
    lambda ex: {"formatted_text": format_alpaca_example(ex)}
)

print("\n--- Sample Formatted Training Sample ---")
print(formatted_dataset[0]["formatted_text"])

## 5. PEFT / LoRA Adapter Configuration
We prepare the 8-bit quantized model for $k$-bit gradient training using `prepare_model_for_kbit_training()` and attach LoRA adapters to all major linear projection layers (`c_attn`, `c_proj`, `c_fc`).

### 🔹 Step 5: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
# 1. Prepare quantized model layers for k-bit training
if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

# 2. Define LoRA Hyperparameters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj", "c_fc"]
)

# 3. Inject LoRA adapters into model
model = get_peft_model(model, lora_config)

print("\n--- Trainable Parameter Statistics ---")
model.print_trainable_parameters()

## 6. Supervised Fine-Tuning with `SFTTrainer` (`trl`)
We use `SFTTrainer` with `packing=True` to concatenate short sequences into 512-token chunks, maximizing GPU throughput and efficiency.

### 🔹 Step 6: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
OUTPUT_DIR = "./gpt2-alpaca-qlora"

# Configure Supervised Fine-Tuning Arguments
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    max_steps=-1,  # Full 3-epoch training
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=25,
    save_strategy="epoch",
    fp16=False,
    bf16=True if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else False,
    gradient_checkpointing=False,
    max_length=512,
    packing=True,  # Packs multiple samples into full 512-length sequences for high efficiency
    report_to="none",
    dataset_text_field="formatted_text"
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer
)

print("Starting QLoRA Fine-Tuning...")
train_result = trainer.train()
print("\nTraining complete!")
print(train_result)

## 7. Save LoRA Adapters & Tokenizer

### 🔹 Step 7: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
SAVE_DIR = "./gpt2-alpaca-qlora"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Fine-tuned LoRA adapters and tokenizer saved to: {SAVE_DIR}")

## 8. Post-Training Inference & Alignment Verification
We test our fine-tuned model using our custom chat template to verify that it now follows user instructions accurately.

### 🔹 Step 8: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
model.eval()

def generate_response_after_training(prompt, max_new_tokens=80):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Evaluate on baseline factual questions and complex instructions
eval_prompts = [
    "What is the capital of France?",
    "What is 2 + 2?",
    "Explain what a computer is.",
    "Explain machine learning in simple terms.",
    "Give me three tips for studying mathematics.",
    "Explain the difference between a CPU and a GPU."
]

print("=" * 60)
print("FINE-TUNED MODEL EVALUATION (AFTER QLORA INSTRUCTION TUNING)")
print("=" * 60)
for prompt in eval_prompts:
    print(f"\n[PROMPT]: {prompt}")
    print(f"[FINE-TUNED RESPONSE]:\n{generate_response_after_training(prompt)}")
    print("-" * 40)